In [5]:
# =============================================================================
# AOSR land cover classification — 3-class scheme, 1985–2025 (5-yearly) — XGBoost
#
#   Forest/Shrub <- Forest + Shrub      (upland woody vegetation)
#   Open water   <- Open water
#   Wetland      <- Bog + Fen + Marsh + Swamp
#
# Same structure as the Random Forest Forest/Shrub script; only the classifier differs.
#
# Workflow
#   1. Training points from ABMI photoplots (7 source classes -> 3 classes)
#   2. LandTrendr-fitted 5-yearly composites (Blue, Green, Red, SWIR1, SWIR2, NBR2)
#   3. XGBoost (XGBClassifier) trained on TRAINING_YEAR, applied to all years
#   4. Area totals + transition matrices (watershed and ABMI wetland boundary)
#   5. Independent validation (ABMI wetlands + reclassified WorldCover)
#   6. Maps, Sankey/alluvial diagrams, percentage trend graphs
#
# To rename a merged class, change UPLAND_LABEL or WETLAND_LABEL only.
# =============================================================================


# =============================================================================
# 0. SETUP
# =============================================================================
import os
import numpy as np
import pandas as pd
import ee
import rasterio
from rasterio.features import geometry_mask
from rasterio.mask import mask as rasterio_mask
from rasterio.transform import rowcol
from pyproj import Transformer
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.colors import ListedColormap, BoundaryNorm
from xgboost import XGBClassifier            # pip install xgboost (preinstalled on Colab)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, cohen_kappa_score)

ee.Authenticate()
ee.Initialize(project='osmgee')

from google.colab import drive
drive.mount('/content/drive')


# =============================================================================
# 1. CONFIGURATION
# =============================================================================
ASSET_PROJECT = 'projects/annual-ccdc-new/assets'
ALL_YEARS     = list(range(1985, 2026, 5))     # 1985, 1990, ..., 2025
TRAINING_YEAR = 2020
CRS           = 'EPSG:32612'
SCALE         = 30
PIXEL_AREA_KM2 = SCALE * SCALE / 1e6

# Composites are the same 6 bands as the 7-class run, so they are reused as-is
COMPOSITE_DIR    = '/content/drive/MyDrive/GEE_exports_7class_composites'
COMPOSITE_PREFIX = 'composite_7class_'
EXPORT_COMPOSITES = False   # set True only if composites are missing on Drive

OUT_DIR = '/content/drive/MyDrive/GEE_exports_3class_ForestShrub_XGBoost'   # separate from earlier runs
FIG_DIR = '/content'
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_TAG = 'xgb'   # added to every output file name so nothing overwrites the RF run
RUN_TAG   = '3class_fs'   # fs = Forest/Shrub merged

# ---- XGBoost settings (same as the earlier 7-class XGBoost run) ------------
XGB_PARAMS = dict(
    n_estimators=100,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1,
)
# The earlier XGBoost run used no class weighting. Set True to mimic the RF's
# class_weight='balanced' (XGBoost has no class_weight, so sample weights are used).
USE_BALANCED_WEIGHTS = False

# ---- Class scheme ----------------------------------------------------------
UPLAND_LABEL  = 'Forest/Shrub'
WETLAND_LABEL = 'Wetland'

CLASS_MERGE = {
    'Forest':     UPLAND_LABEL,
    'Shrub':      UPLAND_LABEL,     # shrub merged with forest (upland woody vegetation)
    'Open water': 'Open water',
    'Bog':        WETLAND_LABEL,
    'Fen':        WETLAND_LABEL,
    'Marsh':      WETLAND_LABEL,
    'Swamp':      WETLAND_LABEL,
}

# Alphabetical order; XGBoost is trained on these integer codes (0, 1, 2)
CLASS_LIST   = sorted(set(CLASS_MERGE.values()))
CLASS_TO_INT = {c: i for i, c in enumerate(CLASS_LIST)}
CLASS_NAMES  = {i: c for c, i in CLASS_TO_INT.items()}
N_CLASSES    = len(CLASS_LIST)

COLOR_BY_NAME = {
    UPLAND_LABEL:  '#2E7D32',
    'Open water':  '#1565C0',
    WETLAND_LABEL: '#8D6E63',
}
CLASS_COLORS = {CLASS_TO_INT[n]: col for n, col in COLOR_BY_NAME.items()}

LT_BANDS      = ['Blue', 'Green', 'Red', 'SWIR1', 'SWIR2']
FEATURE_BANDS = LT_BANDS + ['NBR2']

print('Class encoding:', CLASS_TO_INT)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class encoding: {'Forest/Shrub': 0, 'Open water': 1, 'Wetland': 2}


In [8]:
# =============================================================================
# 2. HELPERS
# =============================================================================
def fc_to_features(fc, page=4000):
    """getInfo() a FeatureCollection in pages, avoiding the 5000-element limit."""
    n = fc.size().getInfo()
    lst = fc.toList(n)
    feats = []
    for start in range(0, n, page):
        feats.extend(ee.List(lst).slice(start, start + page).getInfo())
    return feats


def composite_path(year):
    return f'{COMPOSITE_DIR}/{COMPOSITE_PREFIX}{year}.tif'


def classified_path(year):
    return f'{OUT_DIR}/classified_{RUN_TAG}_{MODEL_TAG}_{year}.tif'


def class_areas(arrays):
    """{year: array of km² per class} from {year: float array with NaN = no data}."""
    return {yr: np.array([np.sum(arr == c) for c in range(N_CLASSES)]) * PIXEL_AREA_KM2
            for yr, arr in arrays.items()}


def transition_matrices(arrays, years):
    """{(y1, y2): N x N km² matrix}, rows = from-class, cols = to-class."""
    out = {}
    for y1, y2 in zip(years[:-1], years[1:]):
        a1, a2 = arrays[y1], arrays[y2]
        valid = ~np.isnan(a1) & ~np.isnan(a2)
        idx = a1[valid].astype(np.int64) * N_CLASSES + a2[valid].astype(np.int64)
        counts = np.bincount(idx, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
        out[(y1, y2)] = counts * PIXEL_AREA_KM2
    return out


def print_area_table(arrays, label):
    areas = class_areas(arrays)
    rows = []
    for yr in sorted(areas):
        total = areas[yr].sum()
        row = {'year': yr, 'total_km2': round(total, 2)}
        for c in range(N_CLASSES):
            row[f'{CLASS_NAMES[c]}_km2'] = round(areas[yr][c], 2)
            row[f'{CLASS_NAMES[c]}_pct'] = round(100 * areas[yr][c] / total, 2) if total else 0
        rows.append(row)
    table = pd.DataFrame(rows)
    print(f'\n=== Class areas: {label} ===')
    print(table.to_string(index=False))
    return table




In [13]:
# =============================================================================
# 3. STUDY AREA AND TRAINING POINTS
# =============================================================================
watersheds     = ee.FeatureCollection('projects/osmgee/assets/OSM_Watersheds')
watershed_geom = watersheds.geometry()

training_source = ee.FeatureCollection(
    'projects/annual-ccdc-new/assets/OSMWetland/ABMI_PhotoPlots_Clipped_Watershed'
)

print('Feature count:', training_source.size().getInfo())
print('Source WetlandTyp values:',
      training_source.aggregate_array('WetlandTyp').distinct().getInfo())
print('Source class distribution (polygons):',
      training_source.aggregate_histogram('WetlandTyp').getInfo())

# Polygon area distribution per source class
areas_and_classes = training_source.reduceColumns(
    reducer=ee.Reducer.toList(2), selectors=['Shape_Area', 'WetlandTyp']
).get('list').getInfo()
df_areas = pd.DataFrame(areas_and_classes, columns=['Shape_Area', 'WetlandTyp'])
df_areas['Class3'] = df_areas['WetlandTyp'].map(CLASS_MERGE)
print(df_areas.groupby('Class3')['Shape_Area'].describe())

# ---- Regular grid points inside each photoplot polygon ---------------------
GRID_SPACING_M = 150
merge_dict = ee.Dictionary(CLASS_MERGE)

def grid_points_for_polygon(feature):
    grid_image = ee.Image.pixelCoordinates(ee.Projection(CRS).atScale(GRID_SPACING_M))
    grid_fc = grid_image.sample(region=feature.geometry(), scale=GRID_SPACING_M, geometries=True)
    wtype = feature.get('WetlandTyp')
    return grid_fc.map(lambda pt: pt.set({
        'WetlandTyp': wtype,
        'Class3': merge_dict.get(wtype, 'Unknown'),
        'POLYGON_ID': feature.get('POLYGON_ID'),
    }))

training_points = training_source.map(grid_points_for_polygon).flatten()

print('Total sample points:', training_points.size().getInfo())
print('Source-class distribution:', training_points.aggregate_histogram('WetlandTyp').getInfo())
print('3-class distribution:', training_points.aggregate_histogram('Class3').getInfo())


# ---- Quick visual check of training points (saved to HTML) -----------------
import folium

watershed_centroid = watershed_geom.centroid(maxError=1).coordinates().getInfo()
m = folium.Map(location=[watershed_centroid[1], watershed_centroid[0]],
               zoom_start=10, tiles='OpenStreetMap')

folium.GeoJson(watershed_geom.getInfo(), name='Watershed boundary',
               style_function=lambda x: {'color': 'black', 'weight': 2, 'fillOpacity': 0}
               ).add_to(m)

for feat in fc_to_features(training_points):
    lon, lat = feat['geometry']['coordinates']
    cls = feat['properties'].get('Class3', '')
    folium.CircleMarker(
        location=[lat, lon],
        radius=3 if cls == 'Open water' else 2,
        color=COLOR_BY_NAME.get(cls, '#999999'),
        fill=True, fillOpacity=0.8,
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save(f'{FIG_DIR}/training_points_{RUN_TAG}_{MODEL_TAG}_map.html')
print(f'Map saved to {FIG_DIR}/training_points_{RUN_TAG}_{MODEL_TAG}_map.html')




Feature count: 442
Source WetlandTyp values: ['Fen', 'Marsh', 'Shrub', 'Open water', 'Swamp', 'Forest', 'Bog']
Source class distribution (polygons): {'Bog': 12, 'Fen': 96, 'Forest': 274, 'Marsh': 3, 'Open water': 5, 'Shrub': 10, 'Swamp': 42}
              count          mean           std          min           25%  \
Class3                                                                       
Forest/Shrub  284.0  9.609226e+04  1.957003e+05   257.197589  19334.533883   
Open water      5.0  4.217620e+06  9.395710e+06  6137.805815  11245.566385   
Wetland       153.0  9.570485e+04  1.255325e+05   166.111626  21748.019589   

                       50%            75%           max  
Class3                                                   
Forest/Shrub  45452.960406   86075.755660  2.109552e+06  
Open water    18768.138358   26777.591665  2.102517e+07  
Wetland       45760.164483  122188.254673  8.288974e+05  
Total sample points: 2797
Source-class distribution: {'Bog': 32, 'Fen': 488, 

In [23]:
# =============================================================================
# 4. LANDTRENDR-FITTED 5-YEARLY COLLECTION
# =============================================================================
def load_and_rename_year(year):
    asset_id = f'{ASSET_PROJECT}/LT_fitted_{year}'
    orig_bands = [f'LT_{b}_{year}' for b in LT_BANDS]
    return (ee.Image(asset_id)
            .select(orig_bands, LT_BANDS)
            .set('year', year)
            .set('system:time_start', ee.Date.fromYMD(year, 6, 1).millis()))

def add_no_nir_indices(image):
    return image.addBands(image.normalizedDifference(['SWIR1', 'SWIR2']).rename('NBR2'))

LT_COLLECTION = (ee.ImageCollection.fromImages([load_and_rename_year(y) for y in ALL_YEARS])
                 .map(add_no_nir_indices))

print('Collection size:', LT_COLLECTION.size().getInfo())
print('Bands:', LT_COLLECTION.first().bandNames().getInfo())




Collection size: 9
Bands: ['Blue', 'Green', 'Red', 'SWIR1', 'SWIR2', 'NBR2']


In [24]:
# =============================================================================
# 5. SAMPLE TRAINING DATA AND TRAIN XGBOOST (3 CLASSES)
# =============================================================================
fitted_image = LT_COLLECTION.filter(ee.Filter.eq('year', TRAINING_YEAR)).first()

sampled_training_data = fitted_image.select(FEATURE_BANDS).sampleRegions(
    collection=training_points,
    properties=['WetlandTyp', 'Class3', 'POLYGON_ID'],
    scale=SCALE,
    tileScale=8,
    geometries=True,
)
print('Sampled feature count:', sampled_training_data.size().getInfo())

df = pd.DataFrame([f['properties'] for f in fc_to_features(sampled_training_data)])
df = df.dropna(subset=FEATURE_BANDS)
df = df[df['Class3'].isin(CLASS_LIST)]

print(df.shape)
print('Source classes:\n', df['WetlandTyp'].value_counts())
print('3-class labels:\n', df['Class3'].value_counts())

# XGBoost needs integer labels (0, 1, 2), not class-name strings
X_train = df[FEATURE_BANDS].to_numpy()
y_train = df['Class3'].map(CLASS_TO_INT).to_numpy()

sample_weight = compute_sample_weight('balanced', y_train) if USE_BALANCED_WEIGHTS else None

xgb_model = XGBClassifier(**XGB_PARAMS)
xgb_model.fit(X_train, y_train, sample_weight=sample_weight)

assert list(xgb_model.classes_) == list(range(N_CLASSES)), xgb_model.classes_
print('Model trained on', len(df), 'samples. Classes:',
      {int(c): CLASS_NAMES[int(c)] for c in xgb_model.classes_})
print('Balanced sample weights:', USE_BALANCED_WEIGHTS)

importances = pd.DataFrame({
    'feature': FEATURE_BANDS,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)
print(importances.to_string(index=False))




Sampled feature count: 2785
(2785, 9)
Source classes:
 WetlandTyp
Forest        1145
Open water     920
Fen            488
Swamp          115
Shrub           74
Bog             32
Marsh           11
Name: count, dtype: int64
3-class labels:
 Class3
Forest/Shrub    1219
Open water       920
Wetland          646
Name: count, dtype: int64
Model trained on 2785 samples. Classes: {0: 'Forest/Shrub', 1: 'Open water', 2: 'Wetland'}
Balanced sample weights: False
feature  importance
  SWIR1    0.586343
  Green    0.136775
  SWIR2    0.099850
   NBR2    0.079798
   Blue    0.049318
    Red    0.047915


In [8]:
# =============================================================================
# 6. COMPOSITES (reuse existing exports; export only what is missing)
# =============================================================================
missing_years = [y for y in ALL_YEARS if not os.path.exists(composite_path(y))]
print('Missing composites:', missing_years if missing_years else 'none')

if missing_years and EXPORT_COMPOSITES:
    for year in missing_years:
        year_bands = (LT_COLLECTION.filter(ee.Filter.eq('year', year)).first()
                      .select(FEATURE_BANDS).clip(watershed_geom))
        task = ee.batch.Export.image.toDrive(
            image=year_bands,
            description=f'{COMPOSITE_PREFIX}{year}',
            folder=os.path.basename(COMPOSITE_DIR),
            fileNamePrefix=f'{COMPOSITE_PREFIX}{year}',
            region=watershed_geom,
            scale=SCALE,
            crs=CRS,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
        )
        task.start()
        print(f'{year}: export started, task ID: {task.id}')
    raise SystemExit('Exports started. Re-run from Section 7 once they finish.')
elif missing_years:
    raise FileNotFoundError(f'Composites missing for {missing_years}. '
                            'Set EXPORT_COMPOSITES = True and re-run this section.')



Missing composites: none


In [25]:
# =============================================================================
# 7. CLASSIFY EVERY YEAR LOCALLY
# =============================================================================
PREDICT_CHUNK = 2_000_000   # pixels per predict() call, to limit memory

classified_arrays = {}

for year in ALL_YEARS:
    with rasterio.open(composite_path(year)) as src:
        image_array = src.read().astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata

    n_bands, height, width = image_array.shape
    pixels = image_array.reshape(n_bands, -1).T

    valid = np.isfinite(pixels).all(axis=1)
    if nodata is not None and np.isfinite(nodata):
        valid &= ~(pixels == nodata).all(axis=1)

    pred = np.full(pixels.shape[0], -1, dtype=np.int8)
    valid_idx = np.flatnonzero(valid)
    for start in range(0, valid_idx.size, PREDICT_CHUNK):
        idx = valid_idx[start:start + PREDICT_CHUNK]
        pred[idx] = xgb_model.predict(pixels[idx]).astype(np.int8)   # already 0/1/2

    classified = pred.reshape(height, width)
    classified_arrays[year] = classified

    out_profile = profile.copy()
    out_profile.update(count=1, dtype='int8', nodata=-1)
    with rasterio.open(classified_path(year), 'w', **out_profile) as dst:
        dst.write(classified, 1)

    vals, counts = np.unique(classified, return_counts=True)
    print(f'{year}: saved.', {CLASS_NAMES.get(int(v), 'NoData'): int(n) for v, n in zip(vals, counts)})


1985: saved. {'NoData': 32250060, 'Forest/Shrub': 1298449, 'Open water': 139103, 'Wetland': 547402}
1990: saved. {'NoData': 32204069, 'Forest/Shrub': 1266083, 'Open water': 167115, 'Wetland': 597747}
1995: saved. {'NoData': 32199113, 'Forest/Shrub': 1236411, 'Open water': 174684, 'Wetland': 624806}
2000: saved. {'NoData': 32199097, 'Forest/Shrub': 1329258, 'Open water': 169917, 'Wetland': 536742}
2005: saved. {'NoData': 32192968, 'Forest/Shrub': 1304904, 'Open water': 172866, 'Wetland': 564276}
2010: saved. {'NoData': 32187610, 'Forest/Shrub': 1258105, 'Open water': 177522, 'Wetland': 611777}
2015: saved. {'NoData': 32186792, 'Forest/Shrub': 1199477, 'Open water': 178659, 'Wetland': 670086}
2020: saved. {'NoData': 32187678, 'Forest/Shrub': 1133792, 'Open water': 179225, 'Wetland': 734319}
2025: saved. {'NoData': 32191015, 'Forest/Shrub': 1113431, 'Open water': 168344, 'Wetland': 762224}


In [26]:
# =============================================================================
# 8. WATERSHED MASK, AREA TOTALS, TRANSITIONS
# =============================================================================
watershed_utm_geojson = watershed_geom.transform(CRS, maxError=1).getInfo()

with rasterio.open(classified_path(ALL_YEARS[0])) as src:
    watershed_mask = geometry_mask([watershed_utm_geojson],
                                   out_shape=(src.height, src.width),
                                   transform=src.transform, invert=True)
print('Pixels inside watershed:', int(watershed_mask.sum()))

watershed_arrays = {}
for yr in ALL_YEARS:
    arr = classified_arrays[yr].astype(np.float32)
    arr[~watershed_mask] = np.nan
    arr[arr == -1] = np.nan
    watershed_arrays[yr] = arr

watershed_table = print_area_table(watershed_arrays, 'watershed')
watershed_table.to_csv(f'{OUT_DIR}/class_areas_watershed_{RUN_TAG}_{MODEL_TAG}.csv', index=False)

watershed_transitions = transition_matrices(watershed_arrays, ALL_YEARS)
for (y1, y2), M in watershed_transitions.items():
    print(f'\nTransitions {y1} -> {y2} (km², rows = from, cols = to)')
    print(pd.DataFrame(M.round(2), index=CLASS_LIST, columns=CLASS_LIST))




Pixels inside watershed: 1958621

=== Class areas: watershed ===
 year  total_km2  Forest/Shrub_km2  Forest/Shrub_pct  Open water_km2  Open water_pct  Wetland_km2  Wetland_pct
 1985    1702.81           1118.97             65.71          122.09            7.17       461.74        27.12
 1990    1743.41           1088.65             62.44          146.90            8.43       507.86        29.13
 1995    1747.82           1062.91             60.81          153.83            8.80       531.07        30.38
 2000    1747.90           1144.81             65.50          149.11            8.53       453.98        25.97
 2005    1753.26           1124.67             64.15          151.77            8.66       476.83        27.20
 2010    1758.05           1086.24             61.79          155.93            8.87       515.89        29.34
 2015    1758.78           1037.48             58.99          156.98            8.93       564.32        32.09
 2020    1758.00            980.98             

In [28]:
# =============================================================================
# 9. VALIDATION POINTS
# =============================================================================
def thin_points_by_distance(points_fc, min_distance):
    sorted_points = points_fc.randomColumn('rand', seed=42).sort('rand')

    def thin_iteration(current, previous):
        current = ee.Feature(current)
        previous = ee.List(previous)
        kept_geoms = ee.FeatureCollection(previous).geometry()
        is_far_enough = ee.Algorithms.If(
            ee.FeatureCollection(previous).size().eq(0),
            True,
            current.geometry().distance(kept_geoms, min_distance).gt(min_distance),
        )
        return ee.Algorithms.If(is_far_enough, previous.add(current), previous)

    thinned = sorted_points.toList(sorted_points.size()).iterate(thin_iteration, ee.List([]))
    return ee.FeatureCollection(ee.List(thinned))


def generate_and_thin_class_points(polygons_fc, class_field, class_value, n_points, min_distance):
    class_geom = polygons_fc.filter(ee.Filter.eq(class_field, class_value)).geometry()
    pts = ee.FeatureCollection.randomPoints(region=class_geom, points=n_points, seed=42)
    pts = pts.map(lambda pt: pt.set('ref_class', class_value))
    return thin_points_by_distance(pts, min_distance)


MIN_VALIDATION_DISTANCE_M = 400

# ---- Wetland classes from ABMI wetlands -----------------------------------
wetlands_source = ee.FeatureCollection('projects/annual-ccdc-new/assets/OSMWetland/Abmi_Wetlands')
print('ABMI wetland classes:', wetlands_source.aggregate_histogram('WetlandCla').getInfo())

val_sets = []
for cls in ['Open water', 'Marsh', 'Bog', 'Fen', 'Swamp']:   # smallest first (iterate timeouts)
    pts = generate_and_thin_class_points(wetlands_source, 'WetlandCla', cls,
                                         n_points=100, min_distance=MIN_VALIDATION_DISTANCE_M)
    print(f'{cls}: thinned count:', pts.size().getInfo())
    val_sets.append(pts)

# ---- Forest and Shrub from reclassified WorldCover --------------------------
worldcover_polygons = ee.FeatureCollection(
    'projects/annual-ccdc-new/assets/OSMWetland/ReclassifiedWorldCover')
print('WorldCover classes:', worldcover_polygons.aggregate_histogram('ClassName').getInfo())

forest_val_points = generate_and_thin_class_points(
    worldcover_polygons, 'ClassName', 'Tree Cover',
    n_points=100, min_distance=MIN_VALIDATION_DISTANCE_M
).map(lambda f: f.set('ref_class', 'Forest'))
print('Forest: thinned count:', forest_val_points.size().getInfo())

shrub_val_points = generate_and_thin_class_points(
    worldcover_polygons, 'ClassName', 'Shrubland/Grassland/Bare or sparse vegetation',
    n_points=100, min_distance=MIN_VALIDATION_DISTANCE_M
).map(lambda f: f.set('ref_class', 'Shrub'))
print('Shrub: thinned count:', shrub_val_points.size().getInfo())

full_validation_points = val_sets[0]
for fc in val_sets[1:] + [forest_val_points, shrub_val_points]:
    full_validation_points = full_validation_points.merge(fc)

# ---- Drop validation points too close to training points --------------------
OVERLAP_CHECK_DISTANCE_M = 150
training_geom = training_points.geometry()

validation_clean = (full_validation_points
    .map(lambda f: f.set('dist_to_training',
                         f.geometry().distance(training_geom, OVERLAP_CHECK_DISTANCE_M)))
    .filter(ee.Filter.gt('dist_to_training', OVERLAP_CHECK_DISTANCE_M)))

print('Validation points before overlap check:', full_validation_points.size().getInfo())
print('Validation points after overlap check:', validation_clean.size().getInfo())

val_df = pd.DataFrame([{
    'lon': f['geometry']['coordinates'][0],
    'lat': f['geometry']['coordinates'][1],
    'ref_class': f['properties'].get('ref_class'),
} for f in fc_to_features(validation_clean)])

val_df['ref_class3'] = val_df['ref_class'].map(CLASS_MERGE)
print('Source reference classes:\n', val_df['ref_class'].value_counts())
print('3-class reference:\n', val_df['ref_class3'].value_counts())

# Wetland pools 4 source classes and Forest/Shrub pools 2, so class sizes differ.
# Set True to subsample every class to the size of the smallest one.
BALANCE_VALIDATION = False
if BALANCE_VALIDATION:
    n_min = val_df['ref_class3'].value_counts().min()
    val_df = (val_df.groupby('ref_class3', group_keys=False)
                    .apply(lambda g: g.sample(n_min, random_state=42)))
    print('Balanced reference:\n', val_df['ref_class3'].value_counts())



ABMI wetland classes: {'Bog': 727, 'Fen': 2170, 'Marsh': 532, 'Open water': 228, 'Swamp': 3197}
Open water: thinned count: 91
Marsh: thinned count: 71
Bog: thinned count: 82
Fen: thinned count: 98
Swamp: thinned count: 96
WorldCover classes: {'Shrubland/Grassland/Bare or sparse vegetation': 1, 'Tree Cover': 1}
Forest: thinned count: 99
Shrub: thinned count: 96
Validation points before overlap check: 633
Validation points after overlap check: 607
Source reference classes:
 ref_class
Fen           98
Forest        97
Swamp         96
Shrub         94
Bog           81
Open water    75
Marsh         66
Name: count, dtype: int64
3-class reference:
 ref_class3
Wetland         341
Forest/Shrub    191
Open water       75
Name: count, dtype: int64


In [30]:
# =============================================================================
# 10. ACCURACY ASSESSMENT (TRAINING_YEAR map)
# =============================================================================
with rasterio.open(classified_path(TRAINING_YEAR)) as src:
    to_raster = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
    val_transform = src.transform
    val_array = src.read(1)

predicted = []
for _, row in val_df.iterrows():
    x, y = to_raster.transform(row['lon'], row['lat'])
    r, c = rowcol(val_transform, x, y)
    if 0 <= r < val_array.shape[0] and 0 <= c < val_array.shape[1]:
        predicted.append(int(val_array[r, c]))
    else:
        predicted.append(-1)

val_df['predicted_num'] = predicted
val_df['predicted_class'] = val_df['predicted_num'].map(CLASS_NAMES)

val_df_clean = val_df[val_df['predicted_num'] != -1].copy()
print(f'Usable validation points: {len(val_df_clean)} / {len(val_df)}')

cm = confusion_matrix(val_df_clean['ref_class3'], val_df_clean['predicted_class'], labels=CLASS_LIST)
cm_df = pd.DataFrame(cm, index=[f'True_{l}' for l in CLASS_LIST],
                     columns=[f'Pred_{l}' for l in CLASS_LIST])
print(cm_df)

print('\nOverall accuracy:', round(accuracy_score(val_df_clean['ref_class3'],
                                                  val_df_clean['predicted_class']), 4))
print('Kappa:', round(cohen_kappa_score(val_df_clean['ref_class3'],
                                        val_df_clean['predicted_class']), 4))
print('\nClassification report (precision = user\'s, recall = producer\'s accuracy):')
print(classification_report(val_df_clean['ref_class3'], val_df_clean['predicted_class'],
                            labels=CLASS_LIST))

# How each original reference class was mapped in the 3-class scheme
print('\nPredicted 3-class label by original reference class:')
print(pd.crosstab(val_df_clean['ref_class'], val_df_clean['predicted_class']))

cm_df.to_csv(f'{OUT_DIR}/confusion_matrix_{RUN_TAG}_{MODEL_TAG}_{TRAINING_YEAR}.csv')
val_df.to_csv(f'{OUT_DIR}/validation_points_{RUN_TAG}_{MODEL_TAG}_{TRAINING_YEAR}.csv', index=False)




Usable validation points: 603 / 607
                   Pred_Forest/Shrub  Pred_Open water  Pred_Wetland
True_Forest/Shrub                102                0            89
True_Open water                    6               62             5
True_Wetland                     110                3           226

Overall accuracy: 0.6468
Kappa: 0.3847

Classification report (precision = user's, recall = producer's accuracy):
              precision    recall  f1-score   support

Forest/Shrub       0.47      0.53      0.50       191
  Open water       0.95      0.85      0.90        73
     Wetland       0.71      0.67      0.69       339

    accuracy                           0.65       603
   macro avg       0.71      0.68      0.69       603
weighted avg       0.66      0.65      0.65       603


Predicted 3-class label by original reference class:
predicted_class  Forest/Shrub  Open water  Wetland
ref_class                                         
Bog                        18           

In [29]:
# =============================================================================
# 11. ABMI WETLAND BOUNDARY CLIP
# =============================================================================
osm_wetland = ee.FeatureCollection('projects/osmgee/assets/OSM_Wetland')
wetland_geom_4326 = shape(osm_wetland.geometry().dissolve().getInfo())

to_utm = Transformer.from_crs('EPSG:4326', CRS, always_xy=True)
wetland_geom_utm = shp_transform(to_utm.transform, wetland_geom_4326)

def get_polygons(geom):
    if geom.geom_type == 'Polygon':
        return [geom]
    if geom.geom_type == 'MultiPolygon':
        return list(geom.geoms)
    if geom.geom_type == 'GeometryCollection':
        out = []
        for g in geom.geoms:
            out.extend(get_polygons(g))
        return out
    return []

wetland_shapes = [mapping(p) for p in get_polygons(wetland_geom_utm)]

clipped_arrays = {}
clipped_transforms = {}
for year in ALL_YEARS:
    with rasterio.open(classified_path(year)) as src:
        clipped, clipped_transform = rasterio_mask(src, wetland_shapes, crop=True, nodata=-1)
    arr = clipped[0].astype(np.float32)
    arr[arr == -1] = np.nan
    clipped_arrays[year] = arr
    clipped_transforms[year] = clipped_transform

wetland_table = print_area_table(clipped_arrays, 'ABMI wetland boundary')
wetland_table.to_csv(f'{OUT_DIR}/class_areas_abmi_wetland_{RUN_TAG}_{MODEL_TAG}.csv', index=False)





=== Class areas: ABMI wetland boundary ===
 year  total_km2  Forest/Shrub_km2  Forest/Shrub_pct  Open water_km2  Open water_pct  Wetland_km2  Wetland_pct
 1985      55.57             19.06             34.30            0.19            0.34        36.32        65.37
 1990      55.58             19.15             34.46            0.14            0.25        36.29        65.29
 1995      55.58             20.06             36.09            0.15            0.26        35.37        63.65
 2000      55.58             21.30             38.34            0.13            0.24        34.14        61.43
 2005      55.57             21.17             38.09            0.13            0.23        34.28        61.68
 2010      55.58             18.24             32.81            0.14            0.25        37.20        66.93
 2015      55.58             15.79             28.42            0.13            0.24        39.65        71.35
 2020      55.58             13.00             23.38            0.13

In [ ]:

# =============================================================================
# 12. PLOTTING FUNCTIONS
# =============================================================================
def legend_patches():
    return [mpatches.Patch(facecolor=CLASS_COLORS[c], edgecolor='black', label=CLASS_NAMES[c])
            for c in range(N_CLASSES)]


def plot_class_maps(arrays, title, out_path, transforms=None):
    cmap = ListedColormap([CLASS_COLORS[c] for c in range(N_CLASSES)])
    cmap.set_bad('white')
    norm = BoundaryNorm([-0.5 + i for i in range(N_CLASSES + 1)], cmap.N)

    years = sorted(arrays)
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    for ax, year in zip(axes.flat, years):
        arr = arrays[year]
        extent = None
        if transforms is not None:
            t = transforms[year]
            h, w = arr.shape
            extent = [t.c, t.c + w * t.a, t.f + h * t.e, t.f]
        ax.imshow(arr, cmap=cmap, norm=norm, interpolation='nearest', extent=extent)
        ax.set_title(str(year), fontsize=15)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in list(axes.flat)[len(years):]:
        ax.axis('off')

    plt.suptitle(title, fontsize=16, y=1.0)
    fig.legend(handles=legend_patches(), loc='lower center', bbox_to_anchor=(0.5, -0.02),
               ncol=N_CLASSES, frameon=False, fontsize=13)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()


def bezier_patch(ax, x0, x1, y0_bot, y0_top, y1_bot, y1_top, color, alpha=0.35):
    xm = (x0 + x1) / 2
    verts = [(x0, y0_bot), (xm, y0_bot), (xm, y1_bot), (x1, y1_bot),
             (x1, y1_top), (xm, y1_top), (xm, y0_top), (x0, y0_top), (x0, y0_bot)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
             Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    ax.add_patch(PathPatch(Path(verts, codes), facecolor=color, alpha=alpha,
                           edgecolor='none', zorder=2))


def plot_sankey(arrays, out_path, pct_threshold=5.0, title=None):
    """Alluvial diagram. Classes below pct_threshold get a callout above the bar."""
    years = sorted(arrays)
    areas = class_areas(arrays)
    totals = {yr: areas[yr].sum() for yr in years}
    trans = transition_matrices(arrays, years)

    max_total = max(totals.values())
    ann_start, ann_gap = 0.015 * max_total, 0.030 * max_total
    bar_width = 0.3
    xpos = {yr: i for i, yr in enumerate(years)}

    fig, ax = plt.subplots(figsize=(20, 12))
    bar_bottoms = {}
    max_small = 0

    for yr in years:
        x = xpos[yr]
        bottom = 0.0
        bar_bottoms[yr] = {}
        small = []
        for c in range(N_CLASSES):
            h = areas[yr][c]
            pct = 100 * h / totals[yr] if totals[yr] else 0
            ax.bar(x, h, bottom=bottom, width=bar_width, color=CLASS_COLORS[c],
                   edgecolor='white', linewidth=0.3, zorder=3)
            if pct >= pct_threshold:
                txt = ax.text(x, bottom + h / 2, f'{pct:.1f}%', ha='center', va='center',
                              fontsize=15, color='black', fontweight='bold', zorder=4)
                txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])
            elif h > 0:
                small.append((c, pct, bottom + h / 2))
            bar_bottoms[yr][c] = bottom
            bottom += h

        max_small = max(max_small, len(small))
        for i, (c, pct, seg_mid) in enumerate(small):
            ax.annotate(f'{CLASS_NAMES[c]}: {pct:.1f}%',
                        xy=(x, seg_mid), xytext=(x, bottom + ann_start + i * ann_gap),
                        fontsize=15, color='black', fontweight='bold', ha='center', va='bottom',
                        arrowprops=dict(arrowstyle='->', color=CLASS_COLORS[c], lw=1.2),
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=CLASS_COLORS[c],
                                  alpha=0.85, linewidth=0.8),
                        annotation_clip=False, zorder=5)

    # Ribbons: offsets always advance so tiny flows never shift the stacking
    min_draw = max_total * 1e-5
    for (y1, y2), M in trans.items():
        x0 = xpos[y1] + bar_width / 2
        x1 = xpos[y2] - bar_width / 2
        src = dict(bar_bottoms[y1])
        dst = dict(bar_bottoms[y2])
        for c1 in range(N_CLASSES):
            for c2 in range(N_CLASSES):
                f = M[c1, c2]
                if f <= 0:
                    continue
                if f >= min_draw:
                    bezier_patch(ax, x0, x1, src[c1], src[c1] + f, dst[c2], dst[c2] + f,
                                 color=CLASS_COLORS[c1])
                src[c1] += f
                dst[c2] += f

    ax.set_xticks(list(xpos.values()))
    ax.set_xticklabels([str(yr) for yr in years], fontsize=15)
    ax.set_xlim(-0.5, len(years) - 0.5)
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, max_total + ann_start + (max_small + 1) * ann_gap)
    if title:
        ax.set_title(title, fontsize=15, pad=12)

    fig.legend(handles=legend_patches(), loc='lower center', bbox_to_anchor=(0.5, 0.02),
               ncol=N_CLASSES, frameon=False, fontsize=15)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()


def plot_trend(arrays, ylabel, title, out_path, break_pct=2.0, low_share=0.5):
    """Percentage trend lines. If any class drops below break_pct, the y-axis
    is stretched below that value (piecewise scale) so small classes stay readable."""
    years = sorted(arrays)
    areas = class_areas(arrays)
    pcts = {c: [100 * areas[yr][c] / areas[yr].sum() if areas[yr].sum() else 0 for yr in years]
            for c in range(N_CLASSES)}

    y_max = min(100.0, np.ceil(max(max(v) for v in pcts.values()) / 10) * 10)
    use_break = min(min(v) for v in pcts.values()) < break_pct

    if use_break:
        def fwd(y):
            y = np.asarray(y, dtype=float)
            return np.where(y <= break_pct, y / break_pct * low_share,
                            low_share + (y - break_pct) / (y_max - break_pct) * (1 - low_share))
        def inv(t):
            t = np.asarray(t, dtype=float)
            return np.where(t <= low_share, t / low_share * break_pct,
                            break_pct + (t - low_share) / (1 - low_share) * (y_max - break_pct))
    else:
        fwd = lambda y: np.asarray(y, dtype=float) / y_max
        inv = lambda t: np.asarray(t, dtype=float) * y_max

    fig, ax = plt.subplots(figsize=(10, 6))
    if use_break:
        ax.set_yscale('function', functions=(fwd, inv))

    for c in range(N_CLASSES):
        ax.plot(years, pcts[c], color=CLASS_COLORS[c], linewidth=2.5,
                marker='o', markersize=6, label=CLASS_NAMES[c])

    if use_break:
        ax.axhline(break_pct, color='grey', linestyle='--', linewidth=1)
        ax.text(years[0] - 0.5, break_pct, f'scale change at {break_pct:g}%',
                fontsize=9, color='grey', va='bottom', ha='left')
        ticks = np.concatenate([np.arange(0, break_pct + 1e-9, break_pct / 4),
                                np.arange(10, y_max + 0.1, 10)])
        ax.set_yticks(ticks)
    else:
        ax.yaxis.set_major_locator(plt.MultipleLocator(10))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:g}%'))
    ax.set_ylim(0, y_max)

    # End-of-line labels, spread so they don't overlap
    MIN_GAP = 0.04
    placed = []
    for c in sorted(range(N_CLASSES), key=lambda c: float(fwd(pcts[c][-1]))):
        t = float(fwd(pcts[c][-1]))
        if placed and t - placed[-1] < MIN_GAP:
            t = placed[-1] + MIN_GAP
        placed.append(t)
        val = pcts[c][-1]
        ax.annotate(f'{CLASS_NAMES[c]} ({val:.2f}%)' if val < 1 else f'{CLASS_NAMES[c]} ({val:.1f}%)',
                    xy=(years[-1], val), xytext=(years[-1] + 2, float(inv(t))),
                    textcoords='data', fontsize=11, color=CLASS_COLORS[c],
                    va='center', fontweight='bold',
                    arrowprops=dict(arrowstyle='-', color=CLASS_COLORS[c], lw=0.8))

    ax.set_xticks(years)
    ax.set_xticklabels([str(yr) for yr in years], fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_xlabel('Year', fontsize=13)
    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlim(years[0] - 1, years[-1] + 15)
    ax.tick_params(axis='y', labelsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.legend(handles=legend_patches(), loc='lower center', bbox_to_anchor=(0.5, -0.04),
               ncol=N_CLASSES, frameon=False, fontsize=13)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.12, right=0.82)
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()



In [ ]:

# =============================================================================
# 13. FIGURES — WATERSHED
# =============================================================================
plot_class_maps(watershed_arrays,
                'Classified Land Cover (Forest/Shrub / Open water / Wetland), 1985–2025 — XGBoost',
                f'{FIG_DIR}/classified_maps_{RUN_TAG}_{MODEL_TAG}_watershed.png')

plot_sankey(watershed_arrays, f'{FIG_DIR}/sankey_{RUN_TAG}_{MODEL_TAG}_watershed.png')

plot_trend(watershed_arrays,
           ylabel='Percentage of Total Watershed Area (%)',
           title='Land Cover Class Area Percentage Trend within Watershed Boundary, 1985–2025',
           out_path=f'{FIG_DIR}/trend_{RUN_TAG}_{MODEL_TAG}_watershed.png')




: 

: 

: 

In [ ]:


# =============================================================================
# 14. FIGURES — ABMI WETLAND BOUNDARY
# =============================================================================
plot_class_maps(clipped_arrays,
                'Classified Land Cover Clipped to ABMI Wetland Boundary, 1985–2025 — XGBoost',
                f'{FIG_DIR}/classified_maps_{RUN_TAG}_{MODEL_TAG}_wetland_clipped.png',
                transforms=clipped_transforms)

plot_sankey(clipped_arrays, f'{FIG_DIR}/sankey_{RUN_TAG}_{MODEL_TAG}_wetland_clipped.png')

plot_trend(clipped_arrays,
           ylabel='Percentage of Total Wetland Area (%)',
           title='Land Cover Class Area Percentage Trend within ABMI Boundary, 1985–2025',
           out_path=f'{FIG_DIR}/trend_{RUN_TAG}_{MODEL_TAG}_wetland_clipped.png')

print('Done. Rasters and CSVs in', OUT_DIR, '| figures in', FIG_DIR)

: 

In [ ]:
# =============================================================================
# AOSR land cover classification — 3-class scheme, 1985–2025 (5-yearly) — XGBoost
#
#   Forest/Shrub <- Forest + Shrub      (upland woody vegetation)
#   Open water   <- Open water
#   Wetland      <- Bog + Fen + Marsh + Swamp
#
# Same structure as the Random Forest Forest/Shrub script; only the classifier differs.
#
# Workflow
#   1. Training points from ABMI photoplots (7 source classes -> 3 classes)
#   2. LandTrendr-fitted 5-yearly composites (Blue, Green, Red, SWIR1, SWIR2, NBR2)
#   3. XGBoost (XGBClassifier) trained on TRAINING_YEAR, applied to all years
#   4. Area totals + transition matrices (watershed and ABMI wetland boundary)
#   5. Independent validation (ABMI wetlands + reclassified WorldCover)
#   6. Maps, Sankey/alluvial diagrams, percentage trend graphs
#
# To rename a merged class, change UPLAND_LABEL or WETLAND_LABEL only.
# =============================================================================


# =============================================================================
# 0. SETUP
# =============================================================================
import os
import numpy as np
import pandas as pd
import ee
import rasterio
from rasterio.features import geometry_mask
from rasterio.mask import mask as rasterio_mask
from rasterio.transform import rowcol
from pyproj import Transformer
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.colors import ListedColormap, BoundaryNorm
from xgboost import XGBClassifier            # pip install xgboost (preinstalled on Colab)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, cohen_kappa_score)

ee.Authenticate()
ee.Initialize(project='osmgee')

from google.colab import drive
drive.mount('/content/drive')


# =============================================================================
# 1. CONFIGURATION
# =============================================================================
ASSET_PROJECT = 'projects/annual-ccdc-new/assets'
ALL_YEARS     = list(range(1985, 2026, 5))     # 1985, 1990, ..., 2025
TRAINING_YEAR = 2020
CRS           = 'EPSG:32612'
SCALE         = 30
PIXEL_AREA_KM2 = SCALE * SCALE / 1e6

# Composites are the same 6 bands as the 7-class run, so they are reused as-is
COMPOSITE_DIR    = '/content/drive/MyDrive/GEE_exports_7class_composites'
COMPOSITE_PREFIX = 'composite_7class_'
EXPORT_COMPOSITES = False   # set True only if composites are missing on Drive

OUT_DIR = '/content/drive/MyDrive/GEE_exports_3class_ForestShrub_XGBoost'   # separate from earlier runs
FIG_DIR = '/content'

# Memory settings (the study area is ~150–300 million pixels per year)
REUSE_CLASSIFIED = True   # load existing classified GeoTIFFs from Drive instead of re-predicting
DISPLAY_MAX_PX   = 1500   # max width/height (pixels) of each map panel; maps are thinned for display only
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_TAG = 'xgb'   # added to every output file name so nothing overwrites the RF run
RUN_TAG   = '3class_fs'   # fs = Forest/Shrub merged

# ---- XGBoost settings (same as the earlier 7-class XGBoost run) ------------
XGB_PARAMS = dict(
    n_estimators=100,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1,
)
# The earlier XGBoost run used no class weighting. Set True to mimic the RF's
# class_weight='balanced' (XGBoost has no class_weight, so sample weights are used).
USE_BALANCED_WEIGHTS = False

# ---- Class scheme ----------------------------------------------------------
UPLAND_LABEL  = 'Forest/Shrub'
WETLAND_LABEL = 'Wetland'

CLASS_MERGE = {
    'Forest':     UPLAND_LABEL,
    'Shrub':      UPLAND_LABEL,     # shrub merged with forest (upland woody vegetation)
    'Open water': 'Open water',
    'Bog':        WETLAND_LABEL,
    'Fen':        WETLAND_LABEL,
    'Marsh':      WETLAND_LABEL,
    'Swamp':      WETLAND_LABEL,
}

# Alphabetical order; XGBoost is trained on these integer codes (0, 1, 2)
CLASS_LIST   = sorted(set(CLASS_MERGE.values()))
CLASS_TO_INT = {c: i for i, c in enumerate(CLASS_LIST)}
CLASS_NAMES  = {i: c for c, i in CLASS_TO_INT.items()}
N_CLASSES    = len(CLASS_LIST)

COLOR_BY_NAME = {
    UPLAND_LABEL:  '#2E7D32',
    'Open water':  '#1565C0',
    WETLAND_LABEL: '#8D6E63',
}
CLASS_COLORS = {CLASS_TO_INT[n]: col for n, col in COLOR_BY_NAME.items()}

LT_BANDS      = ['Blue', 'Green', 'Red', 'SWIR1', 'SWIR2']
FEATURE_BANDS = LT_BANDS + ['NBR2']

print('Class encoding:', CLASS_TO_INT)


# =============================================================================
# 2. HELPERS
# =============================================================================
def fc_to_features(fc, page=4000):
    """getInfo() a FeatureCollection in pages, avoiding the 5000-element limit."""
    n = fc.size().getInfo()
    lst = fc.toList(n)
    feats = []
    for start in range(0, n, page):
        feats.extend(ee.List(lst).slice(start, start + page).getInfo())
    return feats


def composite_path(year):
    return f'{COMPOSITE_DIR}/{COMPOSITE_PREFIX}{year}.tif'


def classified_path(year):
    return f'{OUT_DIR}/classified_{RUN_TAG}_{MODEL_TAG}_{year}.tif'


def class_areas(arrays):
    """{year: array of km² per class} from {year: int8 array with -1 = no data}."""
    return {yr: np.array([np.sum(arr == c) for c in range(N_CLASSES)]) * PIXEL_AREA_KM2
            for yr, arr in arrays.items()}


def transition_matrices(arrays, years):
    """{(y1, y2): N x N km² matrix}, rows = from-class, cols = to-class."""
    out = {}
    for y1, y2 in zip(years[:-1], years[1:]):
        a1, a2 = arrays[y1], arrays[y2]
        counts = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
        # Pair-by-pair boolean counts: far less memory than building an index array
        for c1 in range(N_CLASSES):
            m1 = (a1 == c1)
            for c2 in range(N_CLASSES):
                counts[c1, c2] = np.count_nonzero(m1 & (a2 == c2))
            del m1
        out[(y1, y2)] = counts * PIXEL_AREA_KM2
    return out


def downsample_for_display(arr, max_px=None):
    """Keep every n-th pixel so the longest side is <= max_px (display only, not for areas)."""
    max_px = max_px or DISPLAY_MAX_PX
    step = max(1, int(np.ceil(max(arr.shape) / max_px)))
    return arr[::step, ::step]


def print_area_table(arrays, label):
    areas = class_areas(arrays)
    rows = []
    for yr in sorted(areas):
        total = areas[yr].sum()
        row = {'year': yr, 'total_km2': round(total, 2)}
        for c in range(N_CLASSES):
            row[f'{CLASS_NAMES[c]}_km2'] = round(areas[yr][c], 2)
            row[f'{CLASS_NAMES[c]}_pct'] = round(100 * areas[yr][c] / total, 2) if total else 0
        rows.append(row)
    table = pd.DataFrame(rows)
    print(f'\n=== Class areas: {label} ===')
    print(table.to_string(index=False))
    return table


# =============================================================================
# 3. STUDY AREA AND TRAINING POINTS
# =============================================================================
watersheds     = ee.FeatureCollection('projects/osmgee/assets/OSM_Watersheds')
watershed_geom = watersheds.geometry()

training_source = ee.FeatureCollection(
    'projects/annual-ccdc-new/assets/OSMWetland/ABMI_PhotoPlots_Clipped_Watershed'
)

print('Feature count:', training_source.size().getInfo())
print('Source WetlandTyp values:',
      training_source.aggregate_array('WetlandTyp').distinct().getInfo())
print('Source class distribution (polygons):',
      training_source.aggregate_histogram('WetlandTyp').getInfo())

# Polygon area distribution per source class
areas_and_classes = training_source.reduceColumns(
    reducer=ee.Reducer.toList(2), selectors=['Shape_Area', 'WetlandTyp']
).get('list').getInfo()
df_areas = pd.DataFrame(areas_and_classes, columns=['Shape_Area', 'WetlandTyp'])
df_areas['Class3'] = df_areas['WetlandTyp'].map(CLASS_MERGE)
print(df_areas.groupby('Class3')['Shape_Area'].describe())

# ---- Regular grid points inside each photoplot polygon ---------------------
GRID_SPACING_M = 150
merge_dict = ee.Dictionary(CLASS_MERGE)

def grid_points_for_polygon(feature):
    grid_image = ee.Image.pixelCoordinates(ee.Projection(CRS).atScale(GRID_SPACING_M))
    grid_fc = grid_image.sample(region=feature.geometry(), scale=GRID_SPACING_M, geometries=True)
    wtype = feature.get('WetlandTyp')
    return grid_fc.map(lambda pt: pt.set({
        'WetlandTyp': wtype,
        'Class3': merge_dict.get(wtype, 'Unknown'),
        'POLYGON_ID': feature.get('POLYGON_ID'),
    }))

training_points = training_source.map(grid_points_for_polygon).flatten()

print('Total sample points:', training_points.size().getInfo())
print('Source-class distribution:', training_points.aggregate_histogram('WetlandTyp').getInfo())
print('3-class distribution:', training_points.aggregate_histogram('Class3').getInfo())


# ---- Quick visual check of training points (saved to HTML) -----------------
import folium

watershed_centroid = watershed_geom.centroid(maxError=1).coordinates().getInfo()
m = folium.Map(location=[watershed_centroid[1], watershed_centroid[0]],
               zoom_start=10, tiles='OpenStreetMap')

folium.GeoJson(watershed_geom.getInfo(), name='Watershed boundary',
               style_function=lambda x: {'color': 'black', 'weight': 2, 'fillOpacity': 0}
               ).add_to(m)

for feat in fc_to_features(training_points):
    lon, lat = feat['geometry']['coordinates']
    cls = feat['properties'].get('Class3', '')
    folium.CircleMarker(
        location=[lat, lon],
        radius=3 if cls == 'Open water' else 2,
        color=COLOR_BY_NAME.get(cls, '#999999'),
        fill=True, fillOpacity=0.8,
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save(f'{FIG_DIR}/training_points_{RUN_TAG}_{MODEL_TAG}_map.html')
print(f'Map saved to {FIG_DIR}/training_points_{RUN_TAG}_{MODEL_TAG}_map.html')


# =============================================================================
# 4. LANDTRENDR-FITTED 5-YEARLY COLLECTION
# =============================================================================
def load_and_rename_year(year):
    asset_id = f'{ASSET_PROJECT}/LT_fitted_{year}'
    orig_bands = [f'LT_{b}_{year}' for b in LT_BANDS]
    return (ee.Image(asset_id)
            .select(orig_bands, LT_BANDS)
            .set('year', year)
            .set('system:time_start', ee.Date.fromYMD(year, 6, 1).millis()))

def add_no_nir_indices(image):
    return image.addBands(image.normalizedDifference(['SWIR1', 'SWIR2']).rename('NBR2'))

LT_COLLECTION = (ee.ImageCollection.fromImages([load_and_rename_year(y) for y in ALL_YEARS])
                 .map(add_no_nir_indices))

print('Collection size:', LT_COLLECTION.size().getInfo())
print('Bands:', LT_COLLECTION.first().bandNames().getInfo())


# =============================================================================
# 5. SAMPLE TRAINING DATA AND TRAIN XGBOOST (3 CLASSES)
# =============================================================================
fitted_image = LT_COLLECTION.filter(ee.Filter.eq('year', TRAINING_YEAR)).first()

sampled_training_data = fitted_image.select(FEATURE_BANDS).sampleRegions(
    collection=training_points,
    properties=['WetlandTyp', 'Class3', 'POLYGON_ID'],
    scale=SCALE,
    tileScale=8,
    geometries=True,
)
print('Sampled feature count:', sampled_training_data.size().getInfo())

df = pd.DataFrame([f['properties'] for f in fc_to_features(sampled_training_data)])
df = df.dropna(subset=FEATURE_BANDS)
df = df[df['Class3'].isin(CLASS_LIST)]

print(df.shape)
print('Source classes:\n', df['WetlandTyp'].value_counts())
print('3-class labels:\n', df['Class3'].value_counts())

# XGBoost needs integer labels (0, 1, 2), not class-name strings
X_train = df[FEATURE_BANDS].to_numpy()
y_train = df['Class3'].map(CLASS_TO_INT).to_numpy()

sample_weight = compute_sample_weight('balanced', y_train) if USE_BALANCED_WEIGHTS else None

xgb_model = XGBClassifier(**XGB_PARAMS)
xgb_model.fit(X_train, y_train, sample_weight=sample_weight)

assert list(xgb_model.classes_) == list(range(N_CLASSES)), xgb_model.classes_
print('Model trained on', len(df), 'samples. Classes:',
      {int(c): CLASS_NAMES[int(c)] for c in xgb_model.classes_})
print('Balanced sample weights:', USE_BALANCED_WEIGHTS)

importances = pd.DataFrame({
    'feature': FEATURE_BANDS,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)
print(importances.to_string(index=False))


# =============================================================================
# 6. COMPOSITES (reuse existing exports; export only what is missing)
# =============================================================================
missing_years = [y for y in ALL_YEARS if not os.path.exists(composite_path(y))]
print('Missing composites:', missing_years if missing_years else 'none')

if missing_years and EXPORT_COMPOSITES:
    for year in missing_years:
        year_bands = (LT_COLLECTION.filter(ee.Filter.eq('year', year)).first()
                      .select(FEATURE_BANDS).clip(watershed_geom))
        task = ee.batch.Export.image.toDrive(
            image=year_bands,
            description=f'{COMPOSITE_PREFIX}{year}',
            folder=os.path.basename(COMPOSITE_DIR),
            fileNamePrefix=f'{COMPOSITE_PREFIX}{year}',
            region=watershed_geom,
            scale=SCALE,
            crs=CRS,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
        )
        task.start()
        print(f'{year}: export started, task ID: {task.id}')
    raise SystemExit('Exports started. Re-run from Section 7 once they finish.')
elif missing_years:
    raise FileNotFoundError(f'Composites missing for {missing_years}. '
                            'Set EXPORT_COMPOSITES = True and re-run this section.')

# =============================================================================
# 7. CLASSIFY EVERY YEAR LOCALLY
# =============================================================================
PREDICT_CHUNK = 2_000_000   # pixels per predict() call, to limit memory

classified_arrays = {}

for year in ALL_YEARS:
    if REUSE_CLASSIFIED and os.path.exists(classified_path(year)):
        with rasterio.open(classified_path(year)) as src:
            classified_arrays[year] = src.read(1).astype(np.int8)
        print(f'{year}: loaded existing classified raster from Drive.')
        continue

    with rasterio.open(composite_path(year)) as src:
        image_array = src.read().astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata

    n_bands, height, width = image_array.shape
    pixels = image_array.reshape(n_bands, -1).T

    valid = np.isfinite(pixels).all(axis=1)
    if nodata is not None and np.isfinite(nodata):
        valid &= ~(pixels == nodata).all(axis=1)

    pred = np.full(pixels.shape[0], -1, dtype=np.int8)
    valid_idx = np.flatnonzero(valid)
    for start in range(0, valid_idx.size, PREDICT_CHUNK):
        idx = valid_idx[start:start + PREDICT_CHUNK]
        pred[idx] = xgb_model.predict(pixels[idx]).astype(np.int8)   # already 0/1/2

    classified = pred.reshape(height, width)
    classified_arrays[year] = classified
    del image_array, pixels, valid, valid_idx   # free the 6-band composite before the next year

    out_profile = profile.copy()
    out_profile.update(count=1, dtype='int8', nodata=-1)
    with rasterio.open(classified_path(year), 'w', **out_profile) as dst:
        dst.write(classified, 1)

    vals, counts = np.unique(classified, return_counts=True)
    print(f'{year}: saved.', {CLASS_NAMES.get(int(v), 'NoData'): int(n) for v, n in zip(vals, counts)})
# =============================================================================
# 8. WATERSHED MASK, AREA TOTALS, TRANSITIONS
# =============================================================================
watershed_utm_geojson = watershed_geom.transform(CRS, maxError=1).getInfo()

with rasterio.open(classified_path(ALL_YEARS[0])) as src:
    watershed_mask = geometry_mask([watershed_utm_geojson],
                                   out_shape=(src.height, src.width),
                                   transform=src.transform, invert=True)
print('Pixels inside watershed:', int(watershed_mask.sum()))

# int8 with -1 = no data (a float32 copy would need ~4x the memory)
watershed_arrays = {}
for yr in ALL_YEARS:
    watershed_arrays[yr] = np.where(watershed_mask, classified_arrays[yr], -1).astype(np.int8)

del classified_arrays          # no longer needed; free memory
import gc; gc.collect()

watershed_table = print_area_table(watershed_arrays, 'watershed')
watershed_table.to_csv(f'{OUT_DIR}/class_areas_watershed_{RUN_TAG}_{MODEL_TAG}.csv', index=False)

watershed_transitions = transition_matrices(watershed_arrays, ALL_YEARS)
for (y1, y2), M in watershed_transitions.items():
    print(f'\nTransitions {y1} -> {y2} (km², rows = from, cols = to)')
    print(pd.DataFrame(M.round(2), index=CLASS_LIST, columns=CLASS_LIST))


# =============================================================================
# 9. VALIDATION POINTS
# =============================================================================
def thin_points_by_distance(points_fc, min_distance):
    sorted_points = points_fc.randomColumn('rand', seed=42).sort('rand')

    def thin_iteration(current, previous):
        current = ee.Feature(current)
        previous = ee.List(previous)
        kept_geoms = ee.FeatureCollection(previous).geometry()
        is_far_enough = ee.Algorithms.If(
            ee.FeatureCollection(previous).size().eq(0),
            True,
            current.geometry().distance(kept_geoms, min_distance).gt(min_distance),
        )
        return ee.Algorithms.If(is_far_enough, previous.add(current), previous)

    thinned = sorted_points.toList(sorted_points.size()).iterate(thin_iteration, ee.List([]))
    return ee.FeatureCollection(ee.List(thinned))


def generate_and_thin_class_points(polygons_fc, class_field, class_value, n_points, min_distance):
    class_geom = polygons_fc.filter(ee.Filter.eq(class_field, class_value)).geometry()
    pts = ee.FeatureCollection.randomPoints(region=class_geom, points=n_points, seed=42)
    pts = pts.map(lambda pt: pt.set('ref_class', class_value))
    return thin_points_by_distance(pts, min_distance)


MIN_VALIDATION_DISTANCE_M = 400

# ---- Wetland classes from ABMI wetlands -----------------------------------
wetlands_source = ee.FeatureCollection('projects/annual-ccdc-new/assets/OSMWetland/Abmi_Wetlands')
print('ABMI wetland classes:', wetlands_source.aggregate_histogram('WetlandCla').getInfo())

val_sets = []
for cls in ['Open water', 'Marsh', 'Bog', 'Fen', 'Swamp']:   # smallest first (iterate timeouts)
    pts = generate_and_thin_class_points(wetlands_source, 'WetlandCla', cls,
                                         n_points=100, min_distance=MIN_VALIDATION_DISTANCE_M)
    print(f'{cls}: thinned count:', pts.size().getInfo())
    val_sets.append(pts)

# ---- Forest and Shrub from reclassified WorldCover --------------------------
worldcover_polygons = ee.FeatureCollection(
    'projects/annual-ccdc-new/assets/OSMWetland/ReclassifiedWorldCover')
print('WorldCover classes:', worldcover_polygons.aggregate_histogram('ClassName').getInfo())

forest_val_points = generate_and_thin_class_points(
    worldcover_polygons, 'ClassName', 'Tree Cover',
    n_points=100, min_distance=MIN_VALIDATION_DISTANCE_M
).map(lambda f: f.set('ref_class', 'Forest'))
print('Forest: thinned count:', forest_val_points.size().getInfo())

shrub_val_points = generate_and_thin_class_points(
    worldcover_polygons, 'ClassName', 'Shrubland/Grassland/Bare or sparse vegetation',
    n_points=100, min_distance=MIN_VALIDATION_DISTANCE_M
).map(lambda f: f.set('ref_class', 'Shrub'))
print('Shrub: thinned count:', shrub_val_points.size().getInfo())

full_validation_points = val_sets[0]
for fc in val_sets[1:] + [forest_val_points, shrub_val_points]:
    full_validation_points = full_validation_points.merge(fc)

# ---- Drop validation points too close to training points --------------------
OVERLAP_CHECK_DISTANCE_M = 150
training_geom = training_points.geometry()

validation_clean = (full_validation_points
    .map(lambda f: f.set('dist_to_training',
                         f.geometry().distance(training_geom, OVERLAP_CHECK_DISTANCE_M)))
    .filter(ee.Filter.gt('dist_to_training', OVERLAP_CHECK_DISTANCE_M)))

print('Validation points before overlap check:', full_validation_points.size().getInfo())
print('Validation points after overlap check:', validation_clean.size().getInfo())

val_df = pd.DataFrame([{
    'lon': f['geometry']['coordinates'][0],
    'lat': f['geometry']['coordinates'][1],
    'ref_class': f['properties'].get('ref_class'),
} for f in fc_to_features(validation_clean)])

val_df['ref_class3'] = val_df['ref_class'].map(CLASS_MERGE)
print('Source reference classes:\n', val_df['ref_class'].value_counts())
print('3-class reference:\n', val_df['ref_class3'].value_counts())

# Wetland pools 4 source classes and Forest/Shrub pools 2, so class sizes differ.
# Set True to subsample every class to the size of the smallest one.
BALANCE_VALIDATION = False
if BALANCE_VALIDATION:
    n_min = val_df['ref_class3'].value_counts().min()
    val_df = (val_df.groupby('ref_class3', group_keys=False)
                    .apply(lambda g: g.sample(n_min, random_state=42)))
    print('Balanced reference:\n', val_df['ref_class3'].value_counts())

# =============================================================================
# 10. ACCURACY ASSESSMENT (TRAINING_YEAR map)
# =============================================================================
with rasterio.open(classified_path(TRAINING_YEAR)) as src:
    to_raster = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
    val_transform = src.transform
    val_array = src.read(1)

predicted = []
for _, row in val_df.iterrows():
    x, y = to_raster.transform(row['lon'], row['lat'])
    r, c = rowcol(val_transform, x, y)
    if 0 <= r < val_array.shape[0] and 0 <= c < val_array.shape[1]:
        predicted.append(int(val_array[r, c]))
    else:
        predicted.append(-1)

val_df['predicted_num'] = predicted
val_df['predicted_class'] = val_df['predicted_num'].map(CLASS_NAMES)

val_df_clean = val_df[val_df['predicted_num'] != -1].copy()
print(f'Usable validation points: {len(val_df_clean)} / {len(val_df)}')

cm = confusion_matrix(val_df_clean['ref_class3'], val_df_clean['predicted_class'], labels=CLASS_LIST)
cm_df = pd.DataFrame(cm, index=[f'True_{l}' for l in CLASS_LIST],
                     columns=[f'Pred_{l}' for l in CLASS_LIST])
print(cm_df)

print('\nOverall accuracy:', round(accuracy_score(val_df_clean['ref_class3'],
                                                  val_df_clean['predicted_class']), 4))
print('Kappa:', round(cohen_kappa_score(val_df_clean['ref_class3'],
                                        val_df_clean['predicted_class']), 4))
print('\nClassification report (precision = user\'s, recall = producer\'s accuracy):')
print(classification_report(val_df_clean['ref_class3'], val_df_clean['predicted_class'],
                            labels=CLASS_LIST))

# How each original reference class was mapped in the 3-class scheme
print('\nPredicted 3-class label by original reference class:')
print(pd.crosstab(val_df_clean['ref_class'], val_df_clean['predicted_class']))

cm_df.to_csv(f'{OUT_DIR}/confusion_matrix_{RUN_TAG}_{MODEL_TAG}_{TRAINING_YEAR}.csv')
val_df.to_csv(f'{OUT_DIR}/validation_points_{RUN_TAG}_{MODEL_TAG}_{TRAINING_YEAR}.csv', index=False)


# =============================================================================
# 11. ABMI WETLAND BOUNDARY CLIP
# =============================================================================
osm_wetland = ee.FeatureCollection('projects/osmgee/assets/OSM_Wetland')
wetland_geom_4326 = shape(osm_wetland.geometry().dissolve().getInfo())

to_utm = Transformer.from_crs('EPSG:4326', CRS, always_xy=True)
wetland_geom_utm = shp_transform(to_utm.transform, wetland_geom_4326)

def get_polygons(geom):
    if geom.geom_type == 'Polygon':
        return [geom]
    if geom.geom_type == 'MultiPolygon':
        return list(geom.geoms)
    if geom.geom_type == 'GeometryCollection':
        out = []
        for g in geom.geoms:
            out.extend(get_polygons(g))
        return out
    return []

wetland_shapes = [mapping(p) for p in get_polygons(wetland_geom_utm)]

clipped_arrays = {}
clipped_transforms = {}
for year in ALL_YEARS:
    with rasterio.open(classified_path(year)) as src:
        clipped, clipped_transform = rasterio_mask(src, wetland_shapes, crop=True, nodata=-1)
    clipped_arrays[year] = clipped[0].astype(np.int8)   # -1 = no data
    clipped_transforms[year] = clipped_transform

wetland_table = print_area_table(clipped_arrays, 'ABMI wetland boundary')
wetland_table.to_csv(f'{OUT_DIR}/class_areas_abmi_wetland_{RUN_TAG}_{MODEL_TAG}.csv', index=False)



# =============================================================================
# 12. PLOTTING FUNCTIONS
# =============================================================================
def legend_patches():
    return [mpatches.Patch(facecolor=CLASS_COLORS[c], edgecolor='black', label=CLASS_NAMES[c])
            for c in range(N_CLASSES)]


def plot_class_maps(arrays, title, out_path, transforms=None):
    cmap = ListedColormap([CLASS_COLORS[c] for c in range(N_CLASSES)])
    cmap.set_bad('white')
    norm = BoundaryNorm([-0.5 + i for i in range(N_CLASSES + 1)], cmap.N)

    years = sorted(arrays)
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    for ax, year in zip(axes.flat, years):
        arr = arrays[year]
        extent = None
        if transforms is not None:
            t = transforms[year]
            h, w = arr.shape                      # full-resolution size, so the extent is exact
            extent = [t.c, t.c + w * t.a, t.f + h * t.e, t.f]
        shown = np.ma.masked_less(downsample_for_display(arr), 0)   # -1 = no data -> white
        ax.imshow(shown, cmap=cmap, norm=norm, interpolation='nearest', extent=extent)
        ax.set_title(str(year), fontsize=15)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in list(axes.flat)[len(years):]:
        ax.axis('off')

    plt.suptitle(title, fontsize=16, y=1.0)
    fig.legend(handles=legend_patches(), loc='lower center', bbox_to_anchor=(0.5, -0.02),
               ncol=N_CLASSES, frameon=False, fontsize=13)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()


def bezier_patch(ax, x0, x1, y0_bot, y0_top, y1_bot, y1_top, color, alpha=0.35):
    xm = (x0 + x1) / 2
    verts = [(x0, y0_bot), (xm, y0_bot), (xm, y1_bot), (x1, y1_bot),
             (x1, y1_top), (xm, y1_top), (xm, y0_top), (x0, y0_top), (x0, y0_bot)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
             Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    ax.add_patch(PathPatch(Path(verts, codes), facecolor=color, alpha=alpha,
                           edgecolor='none', zorder=2))


def plot_sankey(arrays, out_path, pct_threshold=5.0, title=None):
    """Alluvial diagram. Classes below pct_threshold get a callout above the bar."""
    years = sorted(arrays)
    areas = class_areas(arrays)
    totals = {yr: areas[yr].sum() for yr in years}
    trans = transition_matrices(arrays, years)

    max_total = max(totals.values())
    ann_start, ann_gap = 0.015 * max_total, 0.030 * max_total
    bar_width = 0.3
    xpos = {yr: i for i, yr in enumerate(years)}

    fig, ax = plt.subplots(figsize=(20, 12))
    bar_bottoms = {}
    max_small = 0

    for yr in years:
        x = xpos[yr]
        bottom = 0.0
        bar_bottoms[yr] = {}
        small = []
        for c in range(N_CLASSES):
            h = areas[yr][c]
            pct = 100 * h / totals[yr] if totals[yr] else 0
            ax.bar(x, h, bottom=bottom, width=bar_width, color=CLASS_COLORS[c],
                   edgecolor='white', linewidth=0.3, zorder=3)
            if pct >= pct_threshold:
                txt = ax.text(x, bottom + h / 2, f'{pct:.1f}%', ha='center', va='center',
                              fontsize=15, color='black', fontweight='bold', zorder=4)
                txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])
            elif h > 0:
                small.append((c, pct, bottom + h / 2))
            bar_bottoms[yr][c] = bottom
            bottom += h

        max_small = max(max_small, len(small))
        for i, (c, pct, seg_mid) in enumerate(small):
            ax.annotate(f'{CLASS_NAMES[c]}: {pct:.1f}%',
                        xy=(x, seg_mid), xytext=(x, bottom + ann_start + i * ann_gap),
                        fontsize=15, color='black', fontweight='bold', ha='center', va='bottom',
                        arrowprops=dict(arrowstyle='->', color=CLASS_COLORS[c], lw=1.2),
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=CLASS_COLORS[c],
                                  alpha=0.85, linewidth=0.8),
                        annotation_clip=False, zorder=5)

    # Ribbons: offsets always advance so tiny flows never shift the stacking
    min_draw = max_total * 1e-5
    for (y1, y2), M in trans.items():
        x0 = xpos[y1] + bar_width / 2
        x1 = xpos[y2] - bar_width / 2
        src = dict(bar_bottoms[y1])
        dst = dict(bar_bottoms[y2])
        for c1 in range(N_CLASSES):
            for c2 in range(N_CLASSES):
                f = M[c1, c2]
                if f <= 0:
                    continue
                if f >= min_draw:
                    bezier_patch(ax, x0, x1, src[c1], src[c1] + f, dst[c2], dst[c2] + f,
                                 color=CLASS_COLORS[c1])
                src[c1] += f
                dst[c2] += f

    ax.set_xticks(list(xpos.values()))
    ax.set_xticklabels([str(yr) for yr in years], fontsize=15)
    ax.set_xlim(-0.5, len(years) - 0.5)
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, max_total + ann_start + (max_small + 1) * ann_gap)
    if title:
        ax.set_title(title, fontsize=15, pad=12)

    fig.legend(handles=legend_patches(), loc='lower center', bbox_to_anchor=(0.5, 0.02),
               ncol=N_CLASSES, frameon=False, fontsize=15)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()


def plot_trend(arrays, ylabel, title, out_path, break_pct=2.0, low_share=0.5):
    """Percentage trend lines. If any class drops below break_pct, the y-axis
    is stretched below that value (piecewise scale) so small classes stay readable."""
    years = sorted(arrays)
    areas = class_areas(arrays)
    pcts = {c: [100 * areas[yr][c] / areas[yr].sum() if areas[yr].sum() else 0 for yr in years]
            for c in range(N_CLASSES)}

    y_max = min(100.0, np.ceil(max(max(v) for v in pcts.values()) / 10) * 10)
    use_break = min(min(v) for v in pcts.values()) < break_pct

    if use_break:
        def fwd(y):
            y = np.asarray(y, dtype=float)
            return np.where(y <= break_pct, y / break_pct * low_share,
                            low_share + (y - break_pct) / (y_max - break_pct) * (1 - low_share))
        def inv(t):
            t = np.asarray(t, dtype=float)
            return np.where(t <= low_share, t / low_share * break_pct,
                            break_pct + (t - low_share) / (1 - low_share) * (y_max - break_pct))
    else:
        fwd = lambda y: np.asarray(y, dtype=float) / y_max
        inv = lambda t: np.asarray(t, dtype=float) * y_max

    fig, ax = plt.subplots(figsize=(10, 6))
    if use_break:
        ax.set_yscale('function', functions=(fwd, inv))

    for c in range(N_CLASSES):
        ax.plot(years, pcts[c], color=CLASS_COLORS[c], linewidth=2.5,
                marker='o', markersize=6, label=CLASS_NAMES[c])

    if use_break:
        ax.axhline(break_pct, color='grey', linestyle='--', linewidth=1)
        ax.text(years[0] - 0.5, break_pct, f'scale change at {break_pct:g}%',
                fontsize=9, color='grey', va='bottom', ha='left')
        ticks = np.concatenate([np.arange(0, break_pct + 1e-9, break_pct / 4),
                                np.arange(10, y_max + 0.1, 10)])
        ax.set_yticks(ticks)
    else:
        ax.yaxis.set_major_locator(plt.MultipleLocator(10))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:g}%'))
    ax.set_ylim(0, y_max)

    # End-of-line labels, spread so they don't overlap
    MIN_GAP = 0.04
    placed = []
    for c in sorted(range(N_CLASSES), key=lambda c: float(fwd(pcts[c][-1]))):
        t = float(fwd(pcts[c][-1]))
        if placed and t - placed[-1] < MIN_GAP:
            t = placed[-1] + MIN_GAP
        placed.append(t)
        val = pcts[c][-1]
        ax.annotate(f'{CLASS_NAMES[c]} ({val:.2f}%)' if val < 1 else f'{CLASS_NAMES[c]} ({val:.1f}%)',
                    xy=(years[-1], val), xytext=(years[-1] + 2, float(inv(t))),
                    textcoords='data', fontsize=11, color=CLASS_COLORS[c],
                    va='center', fontweight='bold',
                    arrowprops=dict(arrowstyle='-', color=CLASS_COLORS[c], lw=0.8))

    ax.set_xticks(years)
    ax.set_xticklabels([str(yr) for yr in years], fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_xlabel('Year', fontsize=13)
    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlim(years[0] - 1, years[-1] + 15)
    ax.tick_params(axis='y', labelsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.legend(handles=legend_patches(), loc='lower center', bbox_to_anchor=(0.5, -0.04),
               ncol=N_CLASSES, frameon=False, fontsize=13)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.12, right=0.82)
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()


# =============================================================================
# 13. FIGURES — WATERSHED
# =============================================================================
plot_class_maps(watershed_arrays,
                'Classified Land Cover (Forest/Shrub / Open water / Wetland), 1985–2025 — XGBoost',
                f'{FIG_DIR}/classified_maps_{RUN_TAG}_{MODEL_TAG}_watershed.png')

plot_sankey(watershed_arrays, f'{FIG_DIR}/sankey_{RUN_TAG}_{MODEL_TAG}_watershed.png')

plot_trend(watershed_arrays,
           ylabel='Percentage of Total Watershed Area (%)',
           title='Land Cover Class Area Percentage Trend within Watershed Boundary, 1985–2025',
           out_path=f'{FIG_DIR}/trend_{RUN_TAG}_{MODEL_TAG}_watershed.png')




# =============================================================================
# 14. FIGURES — ABMI WETLAND BOUNDARY
# =============================================================================
plot_class_maps(clipped_arrays,
                'Classified Land Cover Clipped to ABMI Wetland Boundary, 1985–2025 — XGBoost',
                f'{FIG_DIR}/classified_maps_{RUN_TAG}_{MODEL_TAG}_wetland_clipped.png',
                transforms=clipped_transforms)

plot_sankey(clipped_arrays, f'{FIG_DIR}/sankey_{RUN_TAG}_{MODEL_TAG}_wetland_clipped.png')

plot_trend(clipped_arrays,
           ylabel='Percentage of Total Wetland Area (%)',
           title='Land Cover Class Area Percentage Trend within ABMI Boundary, 1985–2025',
           out_path=f'{FIG_DIR}/trend_{RUN_TAG}_{MODEL_TAG}_wetland_clipped.png')

print('Done. Rasters and CSVs in', OUT_DIR, '| figures in', FIG_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class encoding: {'Forest/Shrub': 0, 'Open water': 1, 'Wetland': 2}
Feature count: 442
Source WetlandTyp values: ['Fen', 'Marsh', 'Shrub', 'Open water', 'Swamp', 'Forest', 'Bog']
Source class distribution (polygons): {'Bog': 12, 'Fen': 96, 'Forest': 274, 'Marsh': 3, 'Open water': 5, 'Shrub': 10, 'Swamp': 42}
              count          mean           std          min           25%  \
Class3                                                                       
Forest/Shrub  284.0  9.609226e+04  1.957003e+05   257.197589  19334.533883   
Open water      5.0  4.217620e+06  9.395710e+06  6137.805815  11245.566385   
Wetland       153.0  9.570485e+04  1.255325e+05   166.111626  21748.019589   

                       50%            75%           max  
Class3                                                   
Forest/Shrub  45452.960406   86075.755660  2.109552e+06